In [1]:
"""
Schema drift simulation for the HUM Labs QC ingestion pipeline (Section 4).
Mock data only, structured to match Protocol Section 3's field list, not a real analyzer export.
No AWS dependencies. Standard library only.
"""

CANONICAL_REQUIRED_FIELDS = {"analyzer_id", "timestamp", "event_type", "QC_Flag"}


def extract_schema_fingerprint(record: dict) -> frozenset:
    return frozenset(record.keys())


def detect_schema_drift(incoming_fingerprint: frozenset, active_fingerprint: frozenset) -> dict:
    added = incoming_fingerprint - active_fingerprint
    removed = active_fingerprint - incoming_fingerprint

    if not added and not removed:
        return {"drift": False, "added_fields": set(), "removed_fields": set(), "breaking": False}

    breaking = bool(removed & CANONICAL_REQUIRED_FIELDS)
    return {"drift": True, "added_fields": added, "removed_fields": removed, "breaking": breaking}


def register_glue_schema_version(registry: dict, analyzer_id: str, new_fingerprint: frozenset) -> dict:
    # Mirrors update_table(): a new version is created, prior versions are kept, not overwritten.
    current_version = registry.get(analyzer_id, {}).get("active_version", 0)
    new_version = current_version + 1
    registry[analyzer_id] = {"active_version": new_version, "active_fingerprint": new_fingerprint}
    return registry[analyzer_id]


def normalize_record(record: dict, drift_result: dict, schema_version: int) -> dict:
    if drift_result["breaking"]:
        missing = sorted(drift_result["removed_fields"] & CANONICAL_REQUIRED_FIELDS)
        return {
            "status": "quarantined",
            "reason": f"post-drift required-field gap: {missing}",
            "schema_version": schema_version,
            "record": record,
        }
    return {"status": "curated", "schema_version": schema_version, "record": record}


def run_scenario(label: str, active_fingerprint: frozenset, registry: dict, record: dict):
    print(f"\n--- {label} ---")
    incoming_fingerprint = extract_schema_fingerprint(record)
    drift_result = detect_schema_drift(incoming_fingerprint, active_fingerprint)

    if not drift_result["drift"]:
        print("No drift detected.")
        return active_fingerprint

    print(f"Drift detected. Added: {sorted(drift_result['added_fields'])}, "
          f"Removed: {sorted(drift_result['removed_fields'])}, Breaking: {drift_result['breaking']}")

    entry = register_glue_schema_version(registry, "analyzer_c", incoming_fingerprint)
    print(f"Registered schema version {entry['active_version']} for analyzer_c.")

    outcome = normalize_record(record, drift_result, entry["active_version"])
    print(f"Record outcome: {outcome['status']}")
    if outcome["status"] == "quarantined":
        print(f"Reason: {outcome['reason']}")

    return entry["active_fingerprint"]


if __name__ == "__main__":
    registry: dict = {}

    baseline_record = {
        "analyzer_id": "analyzer_c",
        "timestamp": "2026-08-01T09:00:00Z",
        "event_type": "qc_pass",
        "QC_Flag": "acceptable",
    }
    active_fingerprint = extract_schema_fingerprint(baseline_record)
    registry["analyzer_c"] = {"active_version": 1, "active_fingerprint": active_fingerprint}
    print("Baseline schema registered as version 1.")

    additive_record = {
        "analyzer_id": "analyzer_c",
        "timestamp": "2026-08-15T09:00:00Z",
        "event_type": "qc_pass",
        "QC_Flag": "acceptable",
        "QC_DriftScore": 0.12,
    }
    active_fingerprint = run_scenario(
        "Scenario 1: QC_DriftScore added (non-breaking)",
        active_fingerprint, registry, additive_record,
    )

    breaking_record = {
        "analyzer_id": "analyzer_c",
        "timestamp": "2026-09-01T09:00:00Z",
        "event_type": "qc_pass",
        "QC_DriftScore": 0.31,
    }
    active_fingerprint = run_scenario(
        "Scenario 2: QC_Flag removed (breaking)",
        active_fingerprint, registry, breaking_record,
    )

    print(f"\nFinal registry state: {registry}")

Baseline schema registered as version 1.

--- Scenario 1: QC_DriftScore added (non-breaking) ---
Drift detected. Added: ['QC_DriftScore'], Removed: [], Breaking: False
Registered schema version 2 for analyzer_c.
Record outcome: curated

--- Scenario 2: QC_Flag removed (breaking) ---
Drift detected. Added: [], Removed: ['QC_Flag'], Breaking: True
Registered schema version 3 for analyzer_c.
Record outcome: quarantined
Reason: post-drift required-field gap: ['QC_Flag']

Final registry state: {'analyzer_c': {'active_version': 3, 'active_fingerprint': frozenset({'QC_DriftScore', 'analyzer_id', 'event_type', 'timestamp'})}}
